# 문항 2. 타입 가드 분기 처리와 제네릭 저장소

## in 연산자 기반 타입 좁히기

`User | Product` 유니온 타입을 `in` 연산자로 좁혀서 안전하게 분기 처리합니다. 이 노트북은 단독으로 실행되도록 문항 1의 타입을 다시 정의합니다.

In [ ]:
// 셀 타입 다시 정의
type UserRole = 'admin' | 'customer' | 'guest';
type Category = 'hamster-doll' | 'cat-doll' | 'otter-doll';

interface BaseEntity {
  id: number;
}

interface User extends BaseEntity {
  name: string;
  email: string;
  role: UserRole;
  isActive: boolean;
}

interface Product extends BaseEntity {
  title: string;
  price: number;
  category: Category;
  stock: number;
  discountRate?: number;
}

function formatWon(amount: number): string {
  return amount.toLocaleString('ko-KR') + '원';
}

In [ ]:
// 샘플 데이터 

const users: User[] = [
  { id: 1, name: '김관리', email: 'admin@test.com', role: 'admin', isActive: true },
  { id: 2, name: '이고객', email: 'lee@test.com', role: 'customer', isActive: true },
];

const products: Product[] = [
  { id: 101, title: '하치와레', price: 80000, category: 'cat-doll', stock: 15, discountRate: 0.1 },
  { id: 102, title: '쿠리만쥬', price: 90000, category: 'otter-doll', stock: 40 },
];

In [ ]:
// renderItemSummary (in 연산자로 좁히기)

function renderItemSummary(item: User | Product): string {
  if ('email' in item) {
    // 이 블록 안에서 item은 User로 좁혀진다.
    return `[회원] ${item.name} (${item.role})`;
  } else {
    // 이 블록 안에서 item은 Product로 좁혀진다.
    return `[상품] ${item.title} ${formatWon(item.price)}`;
  }
}

In [ ]:
// 실행과 결과 확인

users.forEach((u) => console.log(renderItemSummary(u)));
products.forEach((p) => console.log(renderItemSummary(p)));

const mixed: (User | Product)[] = [users[0], products[0], users[1], products[1]];
console.log('--- 섞인 배열 ---');
mixed.forEach((item) => console.log(renderItemSummary(item)));

[회원] 김관리 (admin)


[회원] 이고객 (customer)


[상품] 하치와레 80,000원


[상품] 쿠리만쥬 90,000원


--- 섞인 배열 ---


[회원] 김관리 (admin)


[상품] 하치와레 80,000원


[회원] 이고객 (customer)


[상품] 쿠리만쥬 90,000원


### 가드 없이 접근하면 막히는지 확인

`@ts-expect-error`는 "다음 줄은 오류가 나야 정상"이라는 표시입니다. 셀이 에러 없이 끝나면 컴파일러가 두 가지 위험한 접근을 모두 막았다는 의미입니다.

In [ ]:
// 의도적 오류

function wrongAccess(item: User | Product): string {
  // @ts-expect-error 가드 없이 email에 접근하면 Product에는 없으므로 오류가 나야 정상
  return item.email;
}

function wrongBranch(item: User | Product): string {
  if ('email' in item) {
    // @ts-expect-error User 블록 안에서 Product 전용 title에 접근하면 오류가 나야 정상
    return item.title;
  }
  return item.title;
}

console.log('가드 밖/잘못된 분기 접근 2개를 컴파일러가 막았습니다.');

가드 밖/잘못된 분기 접근 2개를 컴파일러가 막았습니다.


### 모든 케이스를 처리했는지 확인 (exhaustive)

유니온의 모든 경우를 처리하고 나면 남는 타입은 `never`입니다. `never`만 받는 `assertNever`를 마지막에 두면 새 타입이 추가됐을 때 처리를 빼먹으면 컴파일 오류가 납니다.

In [ ]:
// exhaustive 버전

function assertNever(value: never): never {
  throw new Error(`처리되지 않은 케이스: ${JSON.stringify(value)}`);
}

function renderItemSummaryStrict(item: User | Product): string {
  if ('email' in item) {
    return `[회원] ${item.name} (${item.role})`;
  }
  if ('title' in item) {
    return `[상품] ${item.title} ${formatWon(item.price)}`;
  }
  return assertNever(item);
}

console.log(renderItemSummaryStrict(users[0]));
console.log(renderItemSummaryStrict(products[1]));

[회원] 김관리 (admin)


[상품] 쿠리만쥬 90,000원


In [ ]:
// 새 타입을 빼먹으면 컴파일러가 알려준다.

interface Coupon extends BaseEntity {
  code: string;
  amountOff: number;
}

function renderWithCoupon(item: User | Product | Coupon): string {
  if ('email' in item) {
    return `[회원] ${item.name} (${item.role})`;
  }
  if ('title' in item) {
    return `[상품] ${item.title} ${formatWon(item.price)}`;
  }
  // @ts-expect-error Coupon 케이스를 처리하지 않아 item이 never가 아니므로 오류가 나야 정상
  return assertNever(item);
}

console.log('새 타입(Coupon)을 처리하지 않으면 컴파일러가 알려줍니다.');

새 타입(Coupon)을 처리하지 않으면 컴파일러가 알려줍니다.
